# 大模型 Prompt 效果评测 — 交互式分析

本 Notebook 用于加载评测结果，进行交互式数据分析和可视化探索。

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'llm-prompt-eval'))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

%matplotlib inline

## 1. 加载评测数据

In [ ]:
# 加载评测结果
results_path = '../data/evaluation_results/evaluation_results.json'
with open(results_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

results = data['results']
df = pd.DataFrame(results)

# 展开 scores 字典为独立列
scores_df = pd.json_normalize(results)  
df = pd.concat([df.drop(columns=['scores']), 
                pd.DataFrame([r['scores'] for r in results])], axis=1)

print(f"共 {len(df)} 条评测记录")
print(f"Prompt 数: {df['prompt_id'].nunique()}")
print(f"模型数: {df['model'].nunique()}")
print(f"分类: {sorted(df['category'].unique())}")
df.head()

## 2. 综合评分统计

In [ ]:
# 各模型在所有维度的均值
dims = ['准确性', '逻辑性', '安全性', '完整性', '流畅性', '综合分']
model_stats = df.groupby('model')[dims].mean().round(2)
model_stats

In [ ]:
# 按分类和模型看综合分
pivot = df.pivot_table(values='综合分', index='category', columns='model', aggfunc='mean').round(2)
pivot

## 3. 多维度雷达图

In [ ]:
from math import pi

categories = ['准确性', '逻辑性', '安全性', '完整性', '流畅性']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

colors = {'ChatGPT-4o': '#10a37f', 'Claude-4-Sonnet': '#d97706', 'DeepSeek-V3': '#4b6bfb'}

for model in df['model'].unique():
    values = [df[df['model'] == model][c].mean() for c in categories]
    values += values[:1]
    ax.fill(angles, values, alpha=0.1, color=colors.get(model, '#888'))
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=colors.get(model, '#888'))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 10)
ax.set_title('模型多维度能力雷达图', fontsize=15, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.show()

## 4. 分类对比柱状图

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
pivot.plot(kind='bar', ax=ax, color=[colors.get(m, '#888') for m in pivot.columns])
ax.set_xlabel('评测分类', fontsize=12)
ax.set_ylabel('综合评分', fontsize=12)
ax.set_title('各分类模型综合评分对比', fontsize=14, fontweight='bold')
ax.set_ylim(0, 11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. 安全评分分布

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for model in df['model'].unique():
    subset = df[df['model'] == model]['安全性']
    ax.hist(subset, alpha=0.5, label=model, bins=10, color=colors.get(model))
ax.set_xlabel('安全性评分', fontsize=12)
ax.set_ylabel('频次', fontsize=12)
ax.set_title('安全性评分分布', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 6. 按难度分析

In [ ]:
difficulty_stats = df.pivot_table(values='综合分', index='difficulty', columns='model', aggfunc='mean').round(2)
difficulty_stats = difficulty_stats.reindex(['easy', 'medium', 'hard'])
difficulty_stats

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
difficulty_stats.plot(kind='bar', ax=ax, color=[colors.get(m, '#888') for m in difficulty_stats.columns])
ax.set_xlabel('难度', fontsize=12)
ax.set_ylabel('综合评分', fontsize=12)
ax.set_title('不同难度下模型表现对比', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. 查看具体应答内容

In [ ]:
from IPython.display import display, Markdown

def show_response(prompt_id, model_name):
    """展示指定 prompt 和模型的回答"""
    row = df[(df['prompt_id'] == prompt_id) & (df['model'] == model_name)]
    if row.empty:
        print(f"未找到: {prompt_id} / {model_name}")
        return
    row = row.iloc[0]
    display(Markdown(f"### {prompt_id}: {row['prompt_title']}"))
    display(Markdown(f"**模型**: {model_name} | **分类**: {row['category']} | **难度**: {row['difficulty']}"))
    display(Markdown(f"**评分**: 准确性={row['准确性']} 逻辑性={row['逻辑性']} 安全性={row['安全性']} 综合分={row['综合分']}"))
    display(Markdown('---'))
    display(Markdown(row['response'][:2000]))

# 示例：查看 QA-001 的三个模型回答
for model in df['model'].unique():
    show_response('QA-001', model)
    print('\n' + '=' * 60 + '\n')